# 35 · Does the PT coordinate measure position, and only position?

Notebook 13's nonbranching scFates coordinate (SCF13) feeds every positional result in the paper.
Notebook 34 compared **segment-label** contrasts with external data; this notebook tests the
**coordinate** itself. DPT from the same embedding (DPT13) is evaluated alongside at no extra cost.

| Check | Question |
|---|---|
| P1 / P1w | Do genes rise or fall along the coordinate the way directly measured segments say, overall and **within** each reviewed segment? |
| Positive controls | Do genes whose zonation externally differs between species keep that difference on the coordinate? |
| P2 | If we inject a species-by-position effect, does the coordinate move to absorb it? |
| P5 | Does the coordinate move when the same physical sections are thinned to equal read depth? |
| n_map | How uncertain is each structure's position, and is it worse in one species? |
| A3 | Could noisier human positions flatten human curves and create the "human-high early, mouse-high late" split? |
| G6 | Does building and testing on the same counts inflate `T_spatial` (count splitting)? |

**Cohort.** Two control mice and two cortex sections from one male human donor; all male. The
human sections are both healthy cortex, so mouse outer-stripe S3 has no human counterpart.
Specimens, not structures, are replicates; every number here is descriptive.

**Protocol.** Rules, thresholds and gene sets were fixed before any output (workstream-1 plan,
sections C0–C6 and the stage 2a addendum). External data resolve only S1/S2/S3 in mouse and
convoluted PT versus S3 in human, so fine order is checked indirectly (P1w).

## 1 · Folders and fixed settings

Pass `--data-root` / `--results-root` or set `PSEUDOSPACE_DATA_ROOT` / `PSEUDOSPACE_RESULTS_ROOT`.

In [ ]:
import argparse
import json
import math
import os
import sys
from hashlib import sha256
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream12 = results_root / 'pt_pathway_remodeling_scfates'
upstream33 = results_root / 'pt_pathway_registration_sensitivity'
external = data_root / 'external'
output = results_root / 'pt_reconstruction_v2' / 'nb35'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
SAMPLES = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
source_paths = {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                for names in SAMPLES.values() for n in names}
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'cross_species_nephron_scfates.h5ad',
            upstream13 / 'ortholog_map_used.csv', upstream12 / 'gene_statistics.csv',
            upstream33 / 'confident_pathways_with_robustness.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz',
            *source_paths.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '35.coordinate_validation.2'  # Bump when a cached calculation changes.
SEED = 0                     # Notebook 13's seed for PCA, Harmony and scFates.
SCFATES_NODES, TRAJECTORY_DIMS, PT_NEIGHBORS = 30, 5, 30
EARLY_MARKERS, LATE_MARKERS = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_FOLDS = 3
MIN_DETECTION = .10          # Per species, in our PT structures.
ANCHOR_LOOSE, ANCHOR_STRICT = .5, 1.0
INJECTION_SHARE_MOD = 10     # Keep HVGs with sha256 mod 10 == 0.
INJECTION_LOG_AMPLITUDE = math.log(2)  # f(s) = log 2 * (2s - 1): 0.5x to 2x along the PT
ABSORPTION_FLOOR = .02
EQUAL_DEPTH_QUANTILE = .20
N_MAP = 20
NOISE_DRAWS, DOSE_SIGMAS, DOSE_DRAWS = 10, (.025, .05, .10), 3
SPLIT_EPSILON, SPLIT_SEED = .5, 35
N_NULL, PATHWAY_SEED, ALPHA, BASIS_DF = 9999, 12, .05, 6
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'
print('Results folder:', output)

## 2 · Inputs: notebook 13's structures, its pass-2 cache and the original counts

The pass-2 nephron embedding is read from notebook 13's own stage cache, identified by its
recorded logic version, and checked against the saved structure list. Raw counts are rebuilt from
the original specimen matrices for every nephron structure; on the 10,076 genes notebook 13 used
they must equal its cached counts exactly. Every perturbation below acts on these counts once.

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rpy2.robjects as ro
import scanpy as sc
import scFates as scf
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

import pseudospace.anchor_coordinate as anchor_module
import pseudospace.coordinate_evaluation as evaluation_module
from pseudospace.anchor_coordinate import (bulk_segment_profiles, census_segment_profiles,
                                           conserved_anchor_mask, positive_control_masks,
                                           pt_specific_mask, segment_contrast)
from pseudospace.coordinate_evaluation import (absorption_ratio, binomial_thin_to_depth, concordance,
                                               fold_noise_variance, group_rank_correlations,
                                               interpolate_curves, log_normalize, poisson_count_split,
                                               reflect_unit, scale_counts, sha256_folds)
from pseudospace.harmony import run_harmony_rpy2, select_harmony_hvgs_by_condition
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_frame, cached_payload, digest

installed_harmony = str(ro.r('as.character(packageVersion("harmony"))')[0])
if installed_harmony != '2.0.5' or scf.__version__ != '1.2.5':
    raise RuntimeError(f'Need R harmony 2.0.5 and scFates 1.2.5; found {installed_harmony}, {scf.__version__}.')

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r').obs.copy()
saved_neph = ad.read_h5ad(upstream13 / 'cross_species_nephron_scfates.h5ad', backed='r').obs.copy()
key_files = [p for p in sorted((upstream13 / 'stage_cache').glob('pass2_nephron_embedding__*.h5ad.key.json'))
             if "'logic': '13-nephron-pass2-v1'" in json.loads(p.read_text())['params']]
if len(key_files) != 1:
    raise FileNotFoundError(f'Expected one notebook-13 pass-2 cache, found {len(key_files)}; rerun notebook 13.')
neph = ad.read_h5ad(str(key_files[0])[:-len('.key.json')])
if not neph.obs_names.equals(saved_neph.index):
    raise ValueError('Notebook 13 pass-2 cache does not match its saved nephron structures.')
pt_mask = neph.obs.coarse_class.astype(str).eq('PT').to_numpy()
pt_rows = np.flatnonzero(pt_mask)
if not neph.obs_names[pt_rows].equals(saved_pt.index):
    raise ValueError('PT rows differ from notebook 13 saved PT structures.')

rebuilt = rebuild_pt_expression(
    saved_neph[['sample', 'comparison_species', 'shared_pseudospace']].astype(
        {'sample': str, 'comparison_species': str}),
    source_paths, pd.read_csv(upstream13 / 'ortholog_map_used.csv'), target_sum=1e4)
COUNTS = sparse.csr_matrix(rebuilt.layers['counts']).astype(np.int64)
genes_all = pd.Index(rebuilt.var_names)
measured_both = rebuilt.var.measured_in_both_inputs.to_numpy(bool)
cache_genes = pd.Index(neph.var_names)
cache_cols = genes_all.get_indexer(cache_genes)
if (cache_cols < 0).any():
    raise ValueError('Notebook 13 genes missing from the rebuilt panel.')
if abs(COUNTS[:, cache_cols] - sparse.csr_matrix(neph.layers['counts'])).max() != 0:
    raise ValueError('Rebuilt counts differ from notebook 13 cached counts.')
ORIGINAL_HVGS = cache_genes[neph.var.highly_variable_for_harmony.to_numpy(bool)]
# scanpy's PCA masks to var['highly_variable'], which here is pass 1's top-2000 list: notebook 13's
# PCA therefore used the intersection. Those are the genes that actually build the coordinate.
CONSTRUCTION_GENES = cache_genes[(neph.var.highly_variable_for_harmony & neph.var.highly_variable).to_numpy(bool)]
neph_obs = neph.obs.copy()
neph_var = neph.var[['exclude_from_harmony_hvg', 'highly_variable']].copy()
del neph

species_pt = saved_pt.comparison_species.astype(str).to_numpy()
specimen_pt = saved_pt['sample'].astype(str).to_numpy()
segment_pt = saved_pt.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
human_pt = species_pt == 'human'
SCF_SAVED = saved_pt.shared_pseudospace.to_numpy(float)
DPT_SAVED = saved_pt.dpt_pseudospace.to_numpy(float)
print(f'{len(neph_obs):,} nephron structures ({len(pt_rows):,} PT) · {len(genes_all):,} rebuilt genes · '
      f'{len(ORIGINAL_HVGS)} notebook-13 Harmony HVGs, {len(CONSTRUCTION_GENES)} used by its PCA')


def pt_lognorm(counts):
    """Notebook-12 normalization for PT rows: log1p over the measured-in-both total x 1e4."""
    block = counts[pt_rows]
    library = np.asarray(block[:, measured_both].sum(axis=1)).ravel().astype(float)
    if (library <= 0).any():
        raise ValueError('A PT structure has no counts in the shared panel.')
    return log_normalize(block, library), library


EVAL_ORIGINAL, LIBRARY_PT = pt_lognorm(COUNTS)
detection = pd.DataFrame({sp: np.asarray((COUNTS[pt_rows][species_pt == sp] > 0).mean(axis=0)).ravel()
                          for sp in SAMPLES}, index=genes_all)

## 3 · External segment data and the fixed gene sets

Anchors are **selected** on mouse microdissection, mouse snRNA from female donors and human snRNA
donor half A. Everything is **evaluated** on mouse snRNA from male donors, human snRNA donor half B
and rat microdissection. Anchors and notebook 13's six orientation markers never enter an
evaluation set.

| Set | Rule |
|---|---|
| A166-rule / A49-rule anchors | same S3−early sign in all selection datasets, |Δ| ≥ 0.5 / ≥ 1, expressed (≥ 1 log2) in each, measured in both species, detected in ≥ 10% of each species' PT, PT-specific (no non-PT tubular class > 2× the PT mean in either species) |
| E (evaluation) | non-anchor, non-marker, measured in both, detected ≥ 10% in the evaluated species, expressed ≥ 1 in the reference |
| D (positive controls) | opposite sign human vs mouse (|Δ| ≥ 0.5 both), or mouse-zonated (|Δ| ≥ 1) and human-flat (|Δ| < 0.25), evaluation datasets |
| H (conserved held-out) | A49-rule genes that are not among the 611 genes notebook 13's PCA used |

In [ ]:
mouse_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
mouse_micro = bulk_segment_profiles(mouse_bulk.drop_duplicates('Gene_symbol').set_index('Gene_symbol'),
                                    '_PT{seg}_').reindex(genes_all)
rat_bulk = pd.read_csv(external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz', sep='\t')
rat_micro = bulk_segment_profiles(rat_bulk.drop_duplicates('Symbol').set_index('Symbol'), '{seg}_')
rat_micro.index = rat_micro.index.astype(str).str.upper()
rat_micro = rat_micro[~rat_micro.index.duplicated()].reindex(genes_all.str.upper())
rat_micro.index = genes_all


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(genes_all)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
HALF_A, HALF_B = human_donors[0::2], human_donors[1::2]
if HALF_A != ['21_015', 'HCA_51_ref', 'PRECISE-34'] or HALF_B != ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36']:
    raise ValueError(f'Human donor halves differ from the pre-registered split: {HALF_A} / {HALF_B}')


def micro_contrast(frame, late, early):
    return frame[late] - frame[early], frame[[late, early]].max(axis=1)


ext = {  # name -> (contrast, level)
    'select_mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'select_mouse_sn_female': segment_contrast(mouse_sn, sex='female'),
    'select_human_sn_halfA': segment_contrast(human_sn, donors=HALF_A),
    'mouse_sn_male': segment_contrast(mouse_sn, sex='male'),
    'mouse_sn_male_S2mS1': segment_contrast(mouse_sn, late='S2', early='S1', sex='male'),
    'mouse_sn_male_S3mS1': segment_contrast(mouse_sn, late='S3', early='S1', sex='male'),
    'mouse_sn_male_S3mS2': segment_contrast(mouse_sn, late='S3', early='S2', sex='male'),
    'mouse_micro': micro_contrast(mouse_micro, 'S3', 'early'),
    'mouse_micro_S2mS1': micro_contrast(mouse_micro, 'S2', 'S1'),
    'mouse_micro_S3mS1': micro_contrast(mouse_micro, 'S3', 'S1'),
    'mouse_micro_S3mS2': micro_contrast(mouse_micro, 'S3', 'S2'),
    'rat_micro': micro_contrast(rat_micro, 'S3', 'early'),
    'human_sn_halfB': segment_contrast(human_sn, donors=HALF_B),
}
contrast = pd.DataFrame({k: v[0] for k, v in ext.items()})
level = pd.DataFrame({k: v[1] for k, v in ext.items()})

# Position-free PT-specificity: linear-scale class means per specimen, averaged within species.
classes = neph_obs.coarse_class.astype(str).to_numpy()
specimen_neph = neph_obs['sample'].astype(str).to_numpy()
library_neph = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
normalized = sparse.csr_matrix(COUNTS.multiply((1e4 / library_neph)[:, None]))
class_means = {}
for sp, names in SAMPLES.items():
    columns = {}
    for cls in sorted(set(classes)):
        per = [np.asarray(normalized[(classes == cls) & (specimen_neph == n)].mean(axis=0)).ravel()
               for n in names if ((classes == cls) & (specimen_neph == n)).sum() >= 20]
        columns[cls] = np.mean(per, axis=0)
    class_means[sp] = pd.DataFrame(columns, index=genes_all)
del normalized
eligible_anchor = (pd.Series(measured_both, index=genes_all) & detection.mouse.ge(MIN_DETECTION)
                   & detection.human.ge(MIN_DETECTION) & pt_specific_mask(class_means))
selection = ['select_mouse_micro', 'select_mouse_sn_female', 'select_human_sn_halfA']
A_LOOSE = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_LOOSE)
A_STRICT = conserved_anchor_mask(contrast[selection], level[selection], eligible_anchor, threshold=ANCHOR_STRICT)
markers = pd.Series(genes_all.isin(EARLY_MARKERS + LATE_MARKERS), index=genes_all)
anchor_sign = np.sign(contrast['select_human_sn_halfA'])


def evaluation_mask(sp, reference):
    return (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers & detection[sp].ge(MIN_DETECTION)
            & level[reference].ge(1) & contrast[reference].notna())


control_eligible = (pd.Series(measured_both, index=genes_all) & ~A_LOOSE & ~markers
                    & detection.mouse.ge(MIN_DETECTION) & detection.human.ge(MIN_DETECTION)
                    & level['mouse_sn_male'].ge(1))
D_OPPOSITE, D_FLAT = positive_control_masks(contrast['human_sn_halfB'], contrast['mouse_sn_male'],
                                            contrast['mouse_micro'], level['human_sn_halfB'], control_eligible)
H_SET = A_STRICT & ~pd.Series(genes_all.isin(CONSTRUCTION_GENES), index=genes_all)
FOLD = pd.Series(sha256_folds(genes_all, N_FOLDS), index=genes_all)
gene_sets_table = pd.DataFrame({
    'anchor_loose_A166rule': A_LOOSE, 'anchor_strict_A49rule': A_STRICT, 'positive_opposite': D_OPPOSITE,
    'positive_flat': D_FLAT, 'conserved_heldout_H': H_SET, 'fold': FOLD,
    'in_notebook13_construction_genes': genes_all.isin(CONSTRUCTION_GENES), **contrast.add_prefix('contrast_'),
    **detection.add_prefix('detection_')})
gene_sets_table.to_csv(output / 'gene_sets_and_external_contrasts.csv')
set_counts = pd.Series({
    'anchors |Δ|≥0.5 (A166 rule)': int(A_LOOSE.sum()),
    'anchors |Δ|≥0.5, late-high': int((A_LOOSE & anchor_sign.gt(0)).sum()),
    'anchors |Δ|≥1 (A49 rule)': int(A_STRICT.sum()),
    'anchors |Δ|≥1, late-high': int((A_STRICT & anchor_sign.gt(0)).sum()),
    'notebook-13 construction genes (PCA)': len(CONSTRUCTION_GENES),
    'strict anchors among construction genes': int((A_STRICT & genes_all.isin(CONSTRUCTION_GENES)).sum()),
    'positive controls: opposite sign': int(D_OPPOSITE.sum()),
    'positive controls: mouse-zonated, human-flat': int(D_FLAT.sum()),
    'conserved held-out H': int(H_SET.sum()),
    'E mouse (male snRNA reference)': int(evaluation_mask('mouse', 'mouse_sn_male').sum()),
    'E human (snRNA half B reference)': int(evaluation_mask('human', 'human_sn_halfB').sum())}, name='genes')
set_counts.to_csv(output / 'gene_set_counts.csv')
display(set_counts.to_frame())

## 4 · Refitting notebook 13's coordinate

`refit_coordinate` is notebook 13's pass-2 and PT code without changes: species-intersection HVGs
on the nephron (752), PCA 50 on those that are also pass 1's top-2000 HVGs (611; scanpy's PCA
masks to `highly_variable`), Harmony (theta 6), the first 5 Harmony dimensions, a 30-node nonbranching
scFates curve rooted by the marker tip score, and DPT from the same root. It takes raw counts, so
every perturbation goes through the same path. A gene fold is withheld by marking its genes as
ineligible for HVG selection. The unperturbed refit must reproduce the saved coordinate.

In [ ]:
code = digest([NOTEBOOK_LOGIC_VERSION, Path(evaluation_module.__file__), Path(anchor_module.__file__),
               Path(run_harmony_rpy2.__code__.co_filename)])
cache_root = output / 'stage_cache'
BASE_DIGEST = digest(COUNTS)


def refit_coordinate(counts, exclude=(), n_map=0, scanpy_normalization=True):
    """Notebook 13 pass-2 embedding plus PT scFates and DPT, from raw counts.

    ``scanpy_normalization=False`` multiplies by 1e4/library instead of dividing: values change by
    about 1e-16 only, which measures the pipeline's numerical sensitivity.
    """
    block = sparse.csr_matrix(counts[:, cache_cols], dtype=np.float64)
    block.sort_indices()  # notebook 13's matrices have sorted indices; summation order matters to scFates
    block.eliminate_zeros()
    obj = ad.AnnData(X=block, obs=neph_obs.copy(), var=neph_var.copy())
    if scanpy_normalization:
        sc.pp.normalize_total(obj, target_sum=1e4)
        sc.pp.log1p(obj)
    else:
        obj.X = log_normalize(block, np.asarray(block.sum(axis=1)).ravel())
    obj.layers['lognorm'] = obj.X.copy()
    obj.var['exclude_from_harmony_hvg'] = obj.var.exclude_from_harmony_hvg.to_numpy(bool) | cache_genes.isin(list(exclude))
    obj = select_harmony_hvgs_by_condition(obj, group_key='comparison_species', groups=('mouse', 'human'),
                                           mode='intersection', min_mean=.0125, max_mean=3, min_disp=.5)
    hvg = obj[:, obj.var.highly_variable_for_harmony.to_numpy(bool)].copy()
    hvg.X = hvg.layers['lognorm'].copy()
    sc.tl.pca(hvg, n_comps=50, random_state=SEED)
    ro.r(f'set.seed({SEED})')
    hvg = run_harmony_rpy2(hvg, batch_key='sample', n_pcs=50, theta=6, lambda_val=1, max_iter=30, tau=0)
    pt = obj[pt_rows].copy()
    pt.obsm['X_trajectory'] = np.asarray(hvg.obsm['X_harmony'][pt_rows, :TRAJECTORY_DIMS], dtype=float)
    values = pt[:, list(EARLY_MARKERS + LATE_MARKERS)].X
    values = values.toarray() if sparse.issparse(values) else np.asarray(values)
    z = np.empty_like(values, dtype=float)
    for sp in SAMPLES:
        rows = species_pt == sp
        z[rows] = (values[rows] - values[rows].mean(axis=0)) / np.maximum(values[rows].std(axis=0), 1e-6)
    pt.obs['early_tip_score'] = -(z[:, 3:].mean(axis=1) - z[:, :3].mean(axis=1))
    scf.tl.curve(pt, Nodes=SCFATES_NODES, use_rep='X_trajectory', ndims_rep=TRAJECTORY_DIMS, seed=SEED)
    graph = pt.uns['graph']
    degree = np.asarray(graph['B']).astype(bool).sum(axis=0)
    if not (len(graph['tips']) == 2 and len(graph['forks']) == 0 and np.all(degree <= 2)):
        raise ValueError('scFates graph is not one nonbranching path.')
    scf.tl.root(pt, 'early_tip_score', tips_only=True)
    root_node = int(pt.uns['graph']['root'])
    membership = np.asarray(pt.obsm['X_R'][:, root_node]).ravel()
    near_root = np.flatnonzero(membership >= np.quantile(membership, .99))
    root_cell = near_root[np.argmax(pt.obs.early_tip_score.to_numpy()[near_root])]
    scf.tl.pseudotime(pt, n_jobs=2, n_map=1, seed=SEED)
    t = pt.obs.t.to_numpy(float)
    result = {'scfates': (t - t.min()) / np.ptp(t), 'n_hvg': int(hvg.n_vars),
              'hvgs': np.asarray(hvg.var_names, dtype=str),
              'pca_genes': np.asarray(hvg.var_names[hvg.var.highly_variable.to_numpy(bool)], dtype=str)}
    if n_map:
        scf.tl.pseudotime(pt, n_jobs=8, n_map=n_map, seed=SEED)
        maps = pt.uns['pseudotime_list']
        result['maps'] = np.vstack([(maps[str(i)].loc[pt.obs_names, 't'].to_numpy(float) - t.min()) / np.ptp(t)
                                    for i in range(n_map)])
    sc.pp.neighbors(pt, n_neighbors=PT_NEIGHBORS, use_rep='X_trajectory', random_state=SEED,
                    key_added='trajectory_neighbors')
    sc.tl.diffmap(pt, neighbors_key='trajectory_neighbors', random_state=SEED)
    pt.uns['iroot'] = int(root_cell)
    sc.tl.dpt(pt, neighbors_key='trajectory_neighbors')
    d = pt.obs.dpt_pseudotime.to_numpy(float)
    finite = np.isfinite(d)
    dpt = np.full(len(d), np.nan)
    dpt[finite] = (d[finite] - d[finite].min()) / np.ptp(d[finite])
    result['dpt'] = dpt
    return result


def run_refit(name, counts, spec, exclude=(), n_map=0, scanpy_normalization=True):
    return cached_payload(f'refit_{name}', lambda: refit_coordinate(counts, exclude, n_map, scanpy_normalization),
                          root=cache_root,
                          params={'spec': spec, 'exclude': sorted(exclude), 'n_map': n_map, 'seed': SEED,
                                  'scanpy_normalization': scanpy_normalization},
                          inputs={'base_counts': BASE_DIGEST}, code=code)


reference = run_refit('reproduction', COUNTS, 'unperturbed', n_map=N_MAP)
# Guard, every run: the refit is notebook 13.
repro = {'scFates Spearman': spearmanr(reference['scfates'], SCF_SAVED).statistic,
         'scFates max |Δ|': float(np.abs(reference['scfates'] - SCF_SAVED).max()),
         'DPT Spearman': spearmanr(reference['dpt'], DPT_SAVED, nan_policy='omit').statistic,
         'HVGs identical': bool(set(reference['hvgs']) == set(ORIGINAL_HVGS)),
         'PCA genes identical': bool(set(reference['pca_genes']) == set(CONSTRUCTION_GENES))}
display(pd.Series(repro, name='reproduction of notebook 13'))
if (repro['scFates Spearman'] < .999 or repro['DPT Spearman'] < .999 or not repro['HVGs identical']
        or not repro['PCA genes identical']):
    raise ValueError(f'Refit does not reproduce notebook 13: {repro}')
COORDS = {'SCF13': SCF_SAVED, 'DPT13': DPT_SAVED}

# Numerical-noise floor: identical data, normalization arithmetic differing by ~1e-16.
floor_fit = run_refit('numerical_floor', COUNTS, 'multiply instead of divide', scanpy_normalization=False)
floor_rows = []
for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
    for n in np.unique(specimen_pt):
        for seg in ('S1', 'S2', 'S3'):
            m = (specimen_pt == n) & (segment_pt == seg)
            floor_rows.append({'coordinate': label, 'specimen': n, 'segment': seg, 'species': species_pt[m][0],
                               'within-segment rho': spearmanr(floor_fit[key][m], COORDS[label][m], nan_policy='omit').statistic,
                               'median |Δz|': float(np.nanmedian(np.abs(floor_fit[key][m] - COORDS[label][m]))),
                               'max |Δz|': float(np.nanmax(np.abs(floor_fit[key][m] - COORDS[label][m])))})
FLOOR = pd.DataFrame(floor_rows)
FLOOR.to_csv(output / 'numerical_floor.csv', index=False)
display(FLOOR.groupby(['coordinate', 'species'])[['within-segment rho', 'median |Δz|', 'max |Δz|']]
        .agg({'within-segment rho': 'median', 'median |Δz|': 'median', 'max |Δz|': 'max'}).round(4)
        .rename_axis(['numerical floor', 'species']))

## 5 · P1 and P1w · Does the coordinate order genes the way direct segment measurements do?

For every gene, r is its Spearman correlation with the coordinate inside one specimen (P1) or
inside one specimen and reviewed segment (P1w), averaged over the species' two specimens. We then
correlate r across genes with the external contrast. Spearman makes this independent of how the
coordinate is stretched.

**Fold protection.** Genes that built a coordinate could agree with external data by construction.
Each fold's genes are therefore scored only on a coordinate refitted without them. The saved
full-data coordinate is shown as a reference, together with the segment labels (an upper reference
for P1, since notebook 34 used them) and ordering by read depth (a nuisance reference for P1w).

In [ ]:
folds = {k: run_refit(f'fold{k}', COUNTS, f'fold {k} withheld',
                      exclude=tuple(cache_genes[FOLD.loc[cache_genes].to_numpy() == k]))
         for k in range(N_FOLDS)}
EVAL_GENES = genes_all[(evaluation_mask('mouse', 'mouse_sn_male') | evaluation_mask('human', 'human_sn_halfB')
                        | evaluation_mask('mouse', 'mouse_micro') | D_OPPOSITE | D_FLAT | H_SET).to_numpy()]
eval_cols = genes_all.get_indexer(EVAL_GENES)
segment_group = np.where(human_pt, np.where(segment_pt == 'S3', 'S3', 'early'), segment_pt)


def species_r(expression, z, within_segment=False, genes=EVAL_GENES):
    """Gene x species table of specimen-averaged Spearman r (NaN unless defined in both specimens)."""
    groups = np.char.add(np.char.add(specimen_pt.astype(str), '|'), segment_group.astype(str)) if within_segment else specimen_pt
    keep = np.isfinite(z)
    table = group_rank_correlations(expression[keep], z[keep], groups[keep])
    table.index = genes
    if not within_segment:
        return pd.DataFrame({sp: table[names].mean(axis=1, skipna=False) for sp, names in SAMPLES.items()})
    return table


def label_contrast(expression, genes=EVAL_GENES):
    """Segment-label S3 − early (S1+S2) per species, averaged over specimens (notebook 34's contrast)."""
    out = {}
    for sp, names in SAMPLES.items():
        per = []
        for n in names:
            late = np.asarray(expression[(specimen_pt == n) & (segment_pt == 'S3')].mean(axis=0)).ravel()
            early = np.asarray(expression[(specimen_pt == n) & (segment_pt != 'S3')].mean(axis=0)).ravel()
            per.append(late - early)
        out[sp] = np.mean(per, axis=0)
    return pd.DataFrame(out, index=genes)


EVAL_X = EVAL_ORIGINAL[:, eval_cols]
P1_REFERENCES = {'mouse': ['mouse_sn_male', 'mouse_micro', 'rat_micro'], 'human': ['human_sn_halfB']}
P1W_REFERENCES = {  # (species, segment group) -> external contrasts, pre-registered primary first
    ('mouse', 'S1'): ['mouse_micro_S2mS1', 'mouse_sn_male_S2mS1'],
    ('mouse', 'S2'): ['mouse_micro_S3mS1', 'mouse_sn_male_S3mS1'],
    ('mouse', 'S3'): ['mouse_micro_S3mS2', 'mouse_sn_male_S3mS2'],
    ('human', 'early'): ['human_sn_halfB'], ('human', 'S3'): ['human_sn_halfB']}


def p1_table(r, label):
    rows = []
    for sp, refs in P1_REFERENCES.items():
        for ref in refs:
            mask = evaluation_mask(sp, ref).loc[r.index].to_numpy()
            rho, n = concordance(r.loc[mask, sp], contrast.loc[r.index[mask], ref])
            rows.append({'coordinate': label, 'species': sp, 'reference': ref, 'rho': rho, 'genes': n})
    return rows


def p1w_table(table, label):
    rows = []
    for (sp, seg), refs in P1W_REFERENCES.items():
        for ref in refs:
            mask = evaluation_mask(sp, ref).loc[table.index].to_numpy()
            for n in SAMPLES[sp]:
                rho, count = concordance(table.loc[mask, f'{n}|{seg}'], contrast.loc[table.index[mask], ref])
                rows.append({'coordinate': label, 'species': sp, 'segment': seg, 'specimen': n,
                             'reference': ref, 'primary': ref == refs[0], 'rho': rho, 'genes': count})
    return rows


fold_of_eval = FOLD.loc[EVAL_GENES].to_numpy()
p1_rows, p1w_rows, R_FULL, R_PROTECTED, RW_PROTECTED = [], [], {}, {}, {}
for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
    R_FULL[label] = species_r(EVAL_X, COORDS[label])
    p1_rows += p1_table(R_FULL[label], f'{label} full data (reference)')
    p1w_rows += p1w_table(species_r(EVAL_X, COORDS[label], within_segment=True), f'{label} full data (reference)')
    pooled, pooled_w = [], []
    for k, fit in folds.items():
        cols = fold_of_eval == k
        pooled.append(species_r(EVAL_X[:, cols], fit[key], genes=EVAL_GENES[cols]))
        pooled_w.append(species_r(EVAL_X[:, cols], fit[key], within_segment=True, genes=EVAL_GENES[cols]))
    R_PROTECTED[label] = pd.concat(pooled).loc[EVAL_GENES]
    RW_PROTECTED[label] = pd.concat(pooled_w).loc[EVAL_GENES]
    p1_rows += p1_table(R_PROTECTED[label], f'{label} fold-protected')
    p1w_rows += p1w_table(RW_PROTECTED[label], f'{label} fold-protected')
labels_contrast = label_contrast(EVAL_X)
p1_rows += p1_table(labels_contrast, 'segment labels (notebook 34 style)')
p1w_rows += p1w_table(species_r(EVAL_X, np.log(LIBRARY_PT), within_segment=True), 'read depth (nuisance reference)')
P1 = pd.DataFrame(p1_rows)
P1W = pd.DataFrame(p1w_rows)
P1.to_csv(output / 'p1_external_concordance.csv', index=False)
P1W.to_csv(output / 'p1w_within_segment_concordance.csv', index=False)
display(P1.pivot_table(index='coordinate', columns=['species', 'reference'], values='rho').round(3))
P1W_SUMMARY = (P1W.groupby(['coordinate', 'species', 'primary']).rho.mean().unstack(['species', 'primary']))
P1W_SUMMARY.columns = [f'{sp} ({"primary" if p else "secondary"} reference)' for sp, p in P1W_SUMMARY.columns]
display(P1W_SUMMARY.round(3).rename_axis('P1w: mean over specimen × segment'))
display(P1W[P1W.primary].pivot_table(index='coordinate', columns=['species', 'segment', 'specimen'],
                                     values='rho').round(2))

# Gene-fold stability of within-segment order (P3a): every fold pair, every specimen × segment.
stability = []
for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
    for a in range(N_FOLDS):
        for b in range(a + 1, N_FOLDS):
            for n in np.unique(specimen_pt):
                for seg in ('S1', 'S2', 'S3'):
                    m = (specimen_pt == n) & (segment_pt == seg)
                    stability.append({'coordinate': label, 'pair': f'{a}-{b}', 'specimen': n, 'segment': seg,
                                      'rho': spearmanr(folds[a][key][m], folds[b][key][m], nan_policy='omit').statistic})
STABILITY = pd.DataFrame(stability)
STABILITY.to_csv(output / 'p3a_gene_fold_within_segment_agreement.csv', index=False)
display(STABILITY.pivot_table(index='coordinate', columns='segment', values='rho', aggfunc='median').round(3)
        .rename_axis('median within-segment rank agreement between gene folds'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout='constrained')
order = ['SCF13 fold-protected', 'DPT13 fold-protected', 'SCF13 full data (reference)', 'segment labels (notebook 34 style)']
main = P1[P1.reference.isin(['mouse_sn_male', 'human_sn_halfB'])]
for i, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
    values = main[main.species.eq(sp)].set_index('coordinate').reindex(order).rho
    axes[0].barh(np.arange(len(order)) + (i - .5) * .38, values, height=.36, color=color, label=sp)
axes[0].set(yticks=np.arange(len(order)), yticklabels=order, xlabel='Spearman across genes',
            title='P1 · coordinate vs external S3 − early')
axes[0].invert_yaxis()
axes[0].legend(frameon=False, loc='lower left', bbox_to_anchor=(0, -.32), ncol=2)
prim = P1W[P1W.primary & P1W.coordinate.isin(['SCF13 fold-protected', 'DPT13 fold-protected', 'read depth (nuisance reference)'])]
cells = prim.groupby(['coordinate', 'species', 'segment']).rho.mean().reset_index()
labels_w = [f'{sp}\n{seg}' for sp, seg in P1W_REFERENCES]
for i, (coord, color) in enumerate((('SCF13 fold-protected', INK), ('DPT13 fold-protected', '#888888'),
                                    ('read depth (nuisance reference)', '#CCCCCC'))):
    vals = [cells[(cells.coordinate == coord) & (cells.species == sp) & (cells.segment == seg)].rho.mean()
            for sp, seg in P1W_REFERENCES]
    axes[1].bar(np.arange(len(labels_w)) + (i - 1) * .27, vals, width=.25, color=color, label=coord)
axes[1].axhline(0, color=INK, lw=.6)
axes[1].set(xticks=np.arange(len(labels_w)), xticklabels=labels_w, ylabel='Spearman across genes',
            title='P1w · within-segment slope vs adjacent external contrast')
axes[1].legend(frameon=False, fontsize=8)
fig.savefig(figures / 'p1_p1w_external_concordance.pdf', bbox_inches='tight')
fig.savefig(figures / 'p1_p1w_external_concordance.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

## 6 · Positive controls · Do externally species-dependent genes keep their difference?

For genes whose S3 − early contrast externally differs between species, the coordinate-based
interaction r_human − r_mouse should have the sign of the external interaction
Δ_human − Δ_mouse (human snRNA half B minus male mouse snRNA). Notebook 34's label-based
interaction is the reference.

In [ ]:
external_interaction = contrast['human_sn_halfB'] - contrast['mouse_sn_male']
control_rows = []
for subset, mask in (('opposite sign', D_OPPOSITE), ('mouse-zonated, human-flat', D_FLAT),
                     ('all positive controls', D_OPPOSITE | D_FLAT)):
    idx = genes_all[mask.to_numpy()]
    for label, table in (('SCF13 fold-protected', R_PROTECTED['SCF13']), ('DPT13 fold-protected', R_PROTECTED['DPT13']),
                         ('SCF13 full data', R_FULL['SCF13']), ('segment labels', labels_contrast)):
        ours = (table.loc[idx, 'human'] - table.loc[idx, 'mouse'])
        ok = ours.notna()
        control_rows.append({'set': subset, 'coordinate': label, 'genes': int(ok.sum()),
                             'sign agreement': float((np.sign(ours[ok]) == np.sign(external_interaction[idx][ok])).mean()),
                             'Spearman': concordance(ours, external_interaction[idx])[0]})
CONTROLS = pd.DataFrame(control_rows)
CONTROLS.to_csv(output / 'positive_control_recovery.csv', index=False)
display(CONTROLS.pivot_table(index='coordinate', columns='set', values=['sign agreement', 'genes']).round(3))

## 7 · P2 · Does the coordinate absorb an injected species-by-position effect?

In human PT only, we multiply the expected counts of 10% of the genes that built notebook 13's
coordinate (its 611 PCA genes, excluding anchors and markers) by
exp(±log 2 · (2s − 1)), where s is the saved coordinate: half the genes rise from 0.5× to 2× along
the PT and half fall. This is the most adversarial direction for the coordinate, because the
effect lies along its own axis. We refit and ask how much of the injected change in
r_human − r_mouse survives on the refitted coordinate, relative to the unperturbed coordinate.
A ratio of 1 means no absorption.

In [ ]:
def hash_int(gene):
    return int(sha256(str(gene).encode()).hexdigest(), 16)


candidates = [g for g in CONSTRUCTION_GENES if not A_LOOSE.get(g, False) and g not in EARLY_MARKERS + LATE_MARKERS]
injected = sorted([g for g in candidates if hash_int(g) % INJECTION_SHARE_MOD == 0], key=hash_int)
signs = np.where(np.arange(len(injected)) % 2 == 0, 1.0, -1.0)
human_pt_rows = pt_rows[human_pt]
f_s0 = INJECTION_LOG_AMPLITUDE * (2 * SCF_SAVED[human_pt] - 1)
INJECTED_COUNTS = scale_counts(COUNTS, human_pt_rows, genes_all.get_indexer(injected),
                               f_s0[:, None] * signs[None, :], np.random.default_rng(351))
injection_fit = run_refit('injection', INJECTED_COUNTS, {'genes': injected, 'signs': signs.tolist(), 'seed': 351})
injected_eval, _ = pt_lognorm(INJECTED_COUNTS)
inj_cols = genes_all.get_indexer(injected)


def interaction(expression, z, genes):
    r = species_r(expression, z, genes=pd.Index(genes))
    return (r.human - r.mouse).to_numpy()


absorption_rows, displacement_rows = [], []
for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
    original = interaction(EVAL_ORIGINAL[:, inj_cols], COORDS[label], injected)
    frozen = interaction(injected_eval[:, inj_cols], COORDS[label], injected)
    refit = interaction(injected_eval[:, inj_cols], injection_fit[key], injected)
    summary = absorption_ratio(original, frozen, refit, floor=ABSORPTION_FLOOR)
    summary['frozen change in expected direction'] = float(np.mean(np.sign(frozen - original) == signs))
    absorption_rows.append({'coordinate': label, **summary})
    for n in np.unique(specimen_pt):
        for seg in ('S1', 'S2', 'S3'):
            m = (specimen_pt == n) & (segment_pt == seg)
            displacement_rows.append({'coordinate': label, 'specimen': n, 'segment': seg,
                                      'species': species_pt[m][0],
                                      'within-segment rho': spearmanr(injection_fit[key][m], COORDS[label][m],
                                                                      nan_policy='omit').statistic,
                                      'median |Δz|': float(np.nanmedian(np.abs(injection_fit[key][m] - COORDS[label][m])))})
ABSORPTION = pd.DataFrame(absorption_rows)
DISPLACEMENT = pd.DataFrame(displacement_rows)
ABSORPTION.to_csv(output / 'p2_absorption.csv', index=False)
DISPLACEMENT.to_csv(output / 'p2_displacement.csv', index=False)
print(f'{len(injected)} injected genes ({int((signs > 0).sum())} rising, {int((signs < 0).sum())} falling in human)')
display(ABSORPTION.round(3))
display(DISPLACEMENT.groupby(['coordinate', 'species'])[['within-segment rho', 'median |Δz|']].median().round(3))

## 8 · P5 · Does the coordinate move when the same sections are thinned to equal depth?

A physical section cannot move when it is sequenced less. Every structure is binomially thinned to
the pooled PT 20th percentile of library size (shallower structures stay as they are), and the
coordinate is refitted. If the coordinate still correlates with the **original** library size after
equal depth, that association is carried by molecular composition rather than by depth.

In [ ]:
library_all = np.asarray(COUNTS[:, measured_both].sum(axis=1)).ravel().astype(float)
depth_target = float(np.quantile(library_all[pt_rows], EQUAL_DEPTH_QUANTILE))
THINNED = binomial_thin_to_depth(COUNTS, library_all, depth_target, np.random.default_rng(352))
equal_fit = run_refit('equal_depth', THINNED, {'target_quantile': EQUAL_DEPTH_QUANTILE, 'seed': 352})
exposure_rows = []
log_library = np.log(LIBRARY_PT)
for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
    for n in np.unique(specimen_pt):
        for seg in ('S1', 'S2', 'S3'):
            m = (specimen_pt == n) & (segment_pt == seg)
            exposure_rows.append({
                'coordinate': label, 'specimen': n, 'segment': seg, 'species': species_pt[m][0],
                'rho(equal-depth, original)': spearmanr(equal_fit[key][m], COORDS[label][m], nan_policy='omit').statistic,
                'rho(z, log library) before': spearmanr(COORDS[label][m], log_library[m], nan_policy='omit').statistic,
                'rho(z, log library) after': spearmanr(equal_fit[key][m], log_library[m], nan_policy='omit').statistic})
EXPOSURE = pd.DataFrame(exposure_rows)
EXPOSURE.to_csv(output / 'p5_exposure_invariance.csv', index=False)
print(f'Equal-depth target: {depth_target:,.0f} counts; {np.mean(library_all[pt_rows] > depth_target):.0%} of PT thinned')
display(EXPOSURE.pivot_table(index=['coordinate', 'species', 'specimen'], columns='segment',
                             values=['rho(equal-depth, original)', 'rho(z, log library) before',
                                     'rho(z, log library) after']).round(2))
P5_MEDIAN = EXPOSURE.groupby('coordinate')['rho(equal-depth, original)'].median()
display(P5_MEDIAN.round(3).rename('median within-segment agreement').to_frame()
        .assign(failure_threshold=.80, fails=lambda d: d.iloc[:, 0] < .80))

## 9 · Position uncertainty from scFates mappings

scFates can sample each structure's assignment to the curve from its soft membership. Twenty
mappings give a per-structure spread. This is conditional on the fitted curve and is not a
calibrated anatomical interval.

In [ ]:
MAPS = reference['maps']
map_sd = MAPS.std(axis=0, ddof=1)
NMAP = (pd.DataFrame({'species': species_pt, 'specimen': specimen_pt, 'segment': segment_pt, 'sd': map_sd,
                      'var': map_sd ** 2})
        .groupby(['species', 'specimen', 'segment']).agg(mean_sd=('sd', 'mean'), median_sd=('sd', 'median'),
                                                         mean_var=('var', 'mean')).reset_index())
NMAP.to_csv(output / 'nmap_position_uncertainty.csv', index=False)
display(NMAP.round(4))
SIGMA2_NMAP = NMAP.groupby('species').mean_var.mean().to_dict()

## 10 · Notebook 12's inputs, for the attenuation and count-split checks

Rebuild notebook 12's PT structures (common 1–99% coordinate support), gene universe, matching
strata and pathway family exactly as notebooks 31 and 33 do, and check that `T_spatial`
reproduces. The 66 robust and 42 core pathways come from notebook 33.

In [ ]:
from pseudospace.pathway_calibration import normal_matched_q
from pseudospace.pathway_remodeling import (fit_nested_trajectories, gene_covariates, local_pathway_curves,
                                            matched_pathway_tests, matching_strata)
from pseudospace.pathways import build_pathway_membership
import pseudospace.pathway_remodeling as remodeling

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')


def support_rows(z):
    """Notebook 12's rule: keep structures inside every specimen's 1st–99th coordinate percentile."""
    frame = pd.DataFrame({'z': z, 'specimen': specimen_pt})
    q = frame.groupby('specimen').z.quantile([.01, .99]).unstack()
    return np.flatnonzero((z >= q[.01].max()) & (z <= q[.99].min()))


def common_grid(z, specimen, n=61):
    lo = max(z[specimen == s].min() for s in np.unique(specimen))
    hi = min(z[specimen == s].max() for s in np.unique(specimen))
    return np.linspace(lo, hi, n)


KEEP12 = support_rows(SCF_SAVED)
universe = gene_covariates(EVAL_ORIGINAL[KEEP12], SCF_SAVED[KEEP12], human_pt[KEEP12], specimen_pt[KEEP12], genes_all)
universe['eligible'] = measured_both & universe.detection.ge(.02).to_numpy()
UNIVERSE = pd.Index(universe.loc[universe.eligible, 'gene'])
U_COLS = genes_all.get_indexer(UNIVERSE)
STRATA = matching_strata(universe.set_index('gene').loc[UNIVERSE], bins=3)
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()),
    universe.loc[measured_both, 'gene'], ortholog_map=orthologs, library_name=lib, tested=list(UNIVERSE), min_genes=1)
    for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(UNIVERSE) - 1))]
GENE_SETS = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
saved_stats = pd.read_csv(upstream12 / 'gene_statistics.csv', index_col=0)
robustness = pd.read_csv(upstream33 / 'confident_pathways_with_robustness.csv')
ROBUST = robustness.loc[robustness.robustness.eq('robust'), 'pathway_id'].tolist()
CORE = robustness.loc[robustness.core.astype(bool), 'pathway_id'].tolist()


def tspatial_fit(expression_pt, z, rows):
    s = z[rows]
    return fit_nested_trajectories(expression_pt[rows][:, U_COLS], s, human_pt[rows], specimen_pt[rows],
                                   common_grid(s, specimen_pt[rows]), basis_df=BASIS_DF)


pathway_code = digest([code, Path(remodeling.__file__)])


def pathway_calls(t_spatial, name):
    frame = pd.DataFrame({'T_spatial': t_spatial}, index=UNIVERSE)
    result = cached_frame(f'matched_{name}', lambda: matched_pathway_tests(frame, GENE_SETS, STRATA, n_null=N_NULL,
                                                                          seed=PATHWAY_SEED),
                          root=cache_root, params={'n_null': N_NULL, 'seed': PATHWAY_SEED},
                          inputs={'scores': frame, 'sets': GENE_SETS, 'strata': STRATA}, code=pathway_code)
    result = normal_matched_q(result)
    return result.set_index('pathway_id').q_normal.le(ALPHA)


def early_late_split(fit, z, rows):
    grid = common_grid(z[rows], specimen_pt[rows])
    local = local_pathway_curves(fit['z'], list(UNIVERSE), {p: GENE_SETS[p] for p in ROBUST}, grid)
    peak = local.loc[local.groupby('pathway_id').divergence.idxmax()]
    human_high = peak.median_member_z.gt(0)
    return {'human-high pathways': int(human_high.sum()), 'mouse-high pathways': int((~human_high).sum()),
            'median peak, human-high': float(peak.position[human_high].median()),
            'median peak, mouse-high': float(peak.position[~human_high].median()),
            'mouse-high with peak in late half': int(((~human_high) & peak.position.ge(.5)).sum()),
            'human-high with peak in early half': int((human_high & peak.position.lt(.5)).sum())}


FIT12 = tspatial_fit(EVAL_ORIGINAL, SCF_SAVED, KEEP12)
if list(saved_stats.index) != list(UNIVERSE) or not np.allclose(FIT12['T_spatial'], saved_stats.T_spatial, rtol=1e-6):
    raise ValueError('Notebook 12 universe or T_spatial does not reproduce; rerun notebook 12.')
CALLS12 = pathway_calls(FIT12['T_spatial'], 'original')
SPLIT12 = early_late_split(FIT12, SCF_SAVED, KEEP12)
print(f'{len(KEEP12):,} structures · {len(UNIVERSE):,} genes · {len(GENE_SETS):,} pathways · '
      f'{int(CALLS12.sum())} strategy-8 calls · robust called {int(CALLS12.loc[ROBUST].sum())}/66')
display(pd.Series(SPLIT12, name='original split (66 robust pathways)'))

## 11 · A3 · Are human curves flatter than mouse curves for externally conserved genes?

H holds genes whose zonation direction is externally conserved and which did **not** build the
coordinate. If human positions were noisier, their fitted human amplitudes (curve at the S3 median
minus curve at the S1 median, signed by the conserved direction) would be systematically smaller
than the mouse ones, and they would be over-represented among high `T_spatial` genes.
Label-based amplitudes and the external snRNA amplitudes show how much of any difference
exists without the coordinate.

In [ ]:
seg12 = segment_pt[KEEP12]
spec12 = specimen_pt[KEEP12]
s12 = SCF_SAVED[KEEP12]
s_S1 = np.mean([np.median(s12[(spec12 == n) & (seg12 == 'S1')]) for n in np.unique(spec12)])
s_S3 = np.mean([np.median(s12[(spec12 == n) & (seg12 == 'S3')]) for n in np.unique(spec12)])
grid12 = common_grid(s12, spec12)
h_genes = [g for g in genes_all[H_SET.to_numpy()] if g in set(UNIVERSE)]
h_idx = UNIVERSE.get_indexer(h_genes)
sign_h = anchor_sign.loc[h_genes].to_numpy()
amp = {sp: (interpolate_curves(FIT12[sp][h_idx], grid12, [s_S3])[:, 0]
            - interpolate_curves(FIT12[sp][h_idx], grid12, [s_S1])[:, 0]) * sign_h for sp in SAMPLES}
lab_x = EVAL_ORIGINAL[KEEP12][:, genes_all.get_indexer(h_genes)]
lab_amp = {}
for sp, names in SAMPLES.items():
    per = [np.asarray(lab_x[(spec12 == n) & (seg12 == 'S3')].mean(axis=0)).ravel()
           - np.asarray(lab_x[(spec12 == n) & (seg12 == 'S1')].mean(axis=0)).ravel() for n in names]
    lab_amp[sp] = np.mean(per, axis=0) * sign_h
percentile = saved_stats.T_spatial.rank(pct=True)
ext_h = np.abs(contrast.loc[h_genes, 'human_sn_halfB']) < np.abs(contrast.loc[h_genes, 'mouse_sn_male'])
ATTENUATION = pd.DataFrame({'gene': h_genes, 'amp_mouse': amp['mouse'], 'amp_human': amp['human'],
                            'label_amp_mouse': lab_amp['mouse'], 'label_amp_human': lab_amp['human'],
                            'T_spatial_percentile': percentile.loc[h_genes].to_numpy(),
                            'external_human_smaller': ext_h.to_numpy()})
ATTENUATION.to_csv(output / 'a3_conserved_heldout_amplitudes.csv', index=False)
positive = ATTENUATION.amp_mouse.gt(0) & ATTENUATION.label_amp_mouse.gt(0)
A3 = pd.Series({
    'conserved held-out genes in universe': len(h_genes),
    'fraction human amplitude < mouse (coordinate)': float((ATTENUATION.amp_human < ATTENUATION.amp_mouse).mean()),
    'fraction human amplitude < mouse (segment labels)': float((ATTENUATION.label_amp_human < ATTENUATION.label_amp_mouse).mean()),
    'fraction |Δ human| < |Δ mouse| (external snRNA)': float(ATTENUATION.external_human_smaller.mean()),
    'median human/mouse amplitude ratio (coordinate)': float(np.median(ATTENUATION.amp_human[positive] / ATTENUATION.amp_mouse[positive])),
    'median human/mouse amplitude ratio (labels)': float(np.median(ATTENUATION.label_amp_human[positive] / ATTENUATION.label_amp_mouse[positive])),
    'median T_spatial percentile': float(ATTENUATION.T_spatial_percentile.median()),
    'S1 / S3 median positions': f'{s_S1:.3f} / {s_S3:.3f}'})
A3['attenuation flag (≥ 70% or percentile ≥ 75)'] = bool(
    A3['fraction human amplitude < mouse (coordinate)'] >= .70 or A3['median T_spatial percentile'] >= .75)
display(A3.to_frame('value'))

### A3b · How much of the early/late split could unequal positional noise explain?

Gene-fold refits estimate each species' positional noise: two coordinates built from disjoint
gene panels disagree by twice the panel-noise variance. If human positions are noisier, adding the
**excess** noise to mouse positions makes attenuation equal in both species. If the split were an
attenuation artifact, it should shrink under this matching. A dose-response over fixed noise levels
shows how much noise the split can tolerate. The fold estimate captures gene-panel noise only, so
it is a lower bound on positional error; the n_map spread is a second estimate.

In [ ]:
SIGMA2_FOLD, fold_cells = fold_noise_variance({k: f['scfates'] for k, f in folds.items()},
                                              specimen_pt, segment_pt, species_pt)
fold_cells.to_csv(output / 'a3b_fold_noise_by_cell.csv')
sigma_ex = {'gene folds': float(np.sqrt(max(SIGMA2_FOLD['human'] - SIGMA2_FOLD['mouse'], 0))),
            'n_map': float(np.sqrt(max(SIGMA2_NMAP['human'] - SIGMA2_NMAP['mouse'], 0)))}
display(pd.DataFrame({'gene-fold σ': {k: np.sqrt(v) for k, v in SIGMA2_FOLD.items()},
                      'n_map σ': {k: np.sqrt(v) for k, v in SIGMA2_NMAP.items()}}).round(4))
print('Excess human σ:', {k: round(v, 4) for k, v in sigma_ex.items()})


def noise_matched(sigma, draw):
    z = SCF_SAVED.copy()
    rng = np.random.default_rng(1000 + draw + int(round(sigma * 1e4)) * 100)
    mouse = ~human_pt
    z[mouse] = reflect_unit(z[mouse] + rng.normal(0, sigma, mouse.sum()))
    fit = tspatial_fit(EVAL_ORIGINAL, z, KEEP12)
    calls = pathway_calls(fit['T_spatial'], f'noise_{sigma:.4f}_{draw}')
    return {'sigma': sigma, 'draw': draw, **early_late_split(fit, z, KEEP12),
            'T_spatial Spearman vs original': spearmanr(fit['T_spatial'], FIT12['T_spatial']).statistic,
            'robust pathways called': int(calls.loc[ROBUST].sum()), 'core pathways called': int(calls.loc[CORE].sum())}


runs = []
for source, sigma in sigma_ex.items():
    if sigma > 0:
        runs += [{'source': source, **noise_matched(sigma, d)} for d in range(NOISE_DRAWS)]
runs += [{'source': 'dose', **noise_matched(sigma, d)} for sigma in DOSE_SIGMAS for d in range(DOSE_DRAWS)]
NOISE = pd.DataFrame(runs)
NOISE.to_csv(output / 'a3b_noise_matching.csv', index=False)
baseline_gap = SPLIT12['median peak, mouse-high'] - SPLIT12['median peak, human-high']
summary_noise = NOISE.groupby(['source', 'sigma']).mean(numeric_only=True).drop(columns='draw')
summary_noise['peak gap (mouse-high − human-high)'] = (summary_noise['median peak, mouse-high']
                                                       - summary_noise['median peak, human-high'])
summary_noise['explained fraction of peak gap'] = 1 - summary_noise['peak gap (mouse-high − human-high)'] / baseline_gap
summary_noise['explained fraction of mouse-high count'] = 1 - summary_noise['mouse-high pathways'] / SPLIT12['mouse-high pathways']
summary_noise.to_csv(output / 'a3b_noise_matching_summary.csv')
display(summary_noise.round(3))
print(f'Original: {SPLIT12["human-high pathways"]} human-high (median peak {SPLIT12["median peak, human-high"]:.3f}), '
      f'{SPLIT12["mouse-high pathways"]} mouse-high (median peak {SPLIT12["median peak, mouse-high"]:.3f}).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), layout='constrained')
lim = np.nanmax(np.abs(np.r_[ATTENUATION.amp_mouse, ATTENUATION.amp_human])) * 1.08
axes[0].scatter(ATTENUATION.amp_mouse, ATTENUATION.amp_human, s=10, color=INK, alpha=.6, linewidths=0)
axes[0].plot([-lim, lim], [-lim, lim], color='#999999', lw=1, ls='--')
axes[0].set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel='mouse amplitude (S3 − S1, conserved direction)',
            ylabel='human amplitude', title=f'A3 · {len(h_genes)} conserved genes not used by the coordinate')
dose = pd.concat([pd.DataFrame([{'sigma': 0.0, 'mouse-high pathways': SPLIT12['mouse-high pathways'],
                                 'human-high pathways': SPLIT12['human-high pathways']}]),
                  summary_noise.loc['dose'].reset_index()], ignore_index=True)
axes[1].plot(dose.sigma, dose['mouse-high pathways'], marker='o', ms=8, lw=2, color=MOUSE, label='mouse-high at peak')
axes[1].plot(dose.sigma, dose['human-high pathways'], marker='o', ms=8, lw=2, color=HUMAN, label='human-high at peak')
for source, sigma in sigma_ex.items():
    if sigma > 0:
        axes[1].axvline(sigma, color='#999999', lw=1, ls=':')
        axes[1].text(sigma, 1, f' estimated ({source})', fontsize=8, color=INK, rotation=90, va='bottom')
axes[1].set(xlabel='noise SD added to mouse positions', ylabel='robust pathways (of 66)',
            title='A3b · split after equalizing positional noise', ylim=(0, 66))
axes[1].legend(frameon=False)
fig.savefig(figures / 'a3_attenuation.pdf', bbox_inches='tight')
fig.savefig(figures / 'a3_attenuation.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)

## 12 · G6 · Count splitting: does using the same counts twice inflate `T_spatial`?

Each count is split into two halves, A ~ Binomial(x, 0.5) and B = x − A, which are independent
under a Poisson model (overdispersed tubule aggregates make this approximate). Three arms test on B:

| Arm | Coordinate | Independent of B's noise? |
|---|---|---|
| a | refitted on A | yes |
| b | the saved full-data coordinate | no (built with B's counts inside the full data) |
| c | refitted on B | no (fully circular) |

If the coordinate's construction inflated positional statistics, arms b and c would call more
pathways and give larger `T_spatial` than arm a.

In [ ]:
SPLIT_A, SPLIT_B = poisson_count_split(COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))
fit_a = run_refit('split_A', SPLIT_A, {'epsilon': SPLIT_EPSILON, 'seed': SPLIT_SEED, 'half': 'A'})
fit_b = run_refit('split_B', SPLIT_B, {'epsilon': SPLIT_EPSILON, 'seed': SPLIT_SEED, 'half': 'B'})
eval_b, _ = pt_lognorm(SPLIT_B)
split_rows = []
for arm, z in (('a · coordinate from A (independent)', fit_a['scfates']),
               ('b · saved full-data coordinate', SCF_SAVED), ('c · coordinate from B (circular)', fit_b['scfates'])):
    rows = KEEP12 if arm.startswith('b') else support_rows(z)
    fit = tspatial_fit(eval_b, z, rows)
    calls = pathway_calls(fit['T_spatial'], 'split_' + arm[0])
    split_rows.append({'arm': arm, 'structures': len(rows),
                       'coordinate Spearman vs saved': spearmanr(z, SCF_SAVED).statistic,
                       'T_spatial Spearman vs notebook 12': spearmanr(fit['T_spatial'], FIT12['T_spatial']).statistic,
                       'median T_spatial': float(np.median(fit['T_spatial'])),
                       'pathway calls': int(calls.sum()), 'robust called (of 66)': int(calls.loc[ROBUST].sum()),
                       'core called (of 42)': int(calls.loc[CORE].sum())})
COUNT_SPLIT = pd.DataFrame(split_rows)
COUNT_SPLIT.to_csv(output / 'g6_count_split.csv', index=False)
display(COUNT_SPLIT.round(3))
print(f'Full data: {int(CALLS12.sum())} calls, robust {int(CALLS12.loc[ROBUST].sum())}/66, '
      f'core {int(CALLS12.loc[CORE].sum())}/42.')

### Position uncertainty carried into `T_spatial`

Refit notebook 12's model on the first ten scFates mappings and compare gene statistics and
robust-pathway calls with the point coordinate.

In [ ]:
map_rows = []
leading = sorted({g for v in robustness.loc[robustness.robustness.eq('robust'), 'leading_edge_genes'].dropna()
                  for g in v.split(';') if g in set(UNIVERSE)})
lead_idx = UNIVERSE.get_indexer(leading)
top = FIT12['T_spatial'] >= np.quantile(FIT12['T_spatial'], .9)
for i in range(10):
    z = np.clip(MAPS[i], 0, 1)
    fit = tspatial_fit(EVAL_ORIGINAL, z, KEEP12)
    calls = pathway_calls(fit['T_spatial'], f'nmap_{i}')
    map_top = fit['T_spatial'] >= np.quantile(fit['T_spatial'], .9)
    map_rows.append({'map': i, 'T_spatial Spearman': spearmanr(fit['T_spatial'], FIT12['T_spatial']).statistic,
                     'leading-edge genes Spearman': spearmanr(fit['T_spatial'][lead_idx], FIT12['T_spatial'][lead_idx]).statistic,
                     'top-10% overlap': float((map_top & top).sum() / top.sum()),
                     'robust called (of 66)': int(calls.loc[ROBUST].sum()), 'core called (of 42)': int(calls.loc[CORE].sum())})
MAP_STATS = pd.DataFrame(map_rows)
MAP_STATS.to_csv(output / 'nmap_tspatial_stability.csv', index=False)
display(MAP_STATS.describe().loc[['mean', 'min', 'max']].round(3))

### Species registration across refits (descriptive, added after the refits were run)

P1 and P1w are within-species. The cross-species question is whether the two species' reviewed
segments stay at the same place on the shared axis when the data change in ways that cannot move
anatomy: a third of the construction genes withheld, half the reads (count split), equal depth,
or the injected effect. The table gives each specimen's median coordinate per reviewed segment and
the largest human-minus-mouse gap. This was not a pre-registered metric.

In [ ]:
refits = {'saved': {'scfates': SCF_SAVED, 'dpt': DPT_SAVED}, 'numerical floor': floor_fit,
          **{f'fold {k} withheld': f for k, f in folds.items()}, 'equal depth': equal_fit,
          'injection': injection_fit, 'count split A': fit_a, 'count split B': fit_b}
registration_rows = []
for name, fit in refits.items():
    for label, key in (('SCF13', 'scfates'), ('DPT13', 'dpt')):
        frame = pd.DataFrame({'z': fit[key], 'species': species_pt, 'segment': segment_pt, 'specimen': specimen_pt})
        medians = frame.groupby(['species', 'specimen', 'segment']).z.median().groupby(['species', 'segment']).mean().unstack()
        gap = medians.loc['human'] - medians.loc['mouse']
        registration_rows.append({'refit': name, 'coordinate': label,
                                  **{f'{seg} human−mouse': gap[seg] for seg in ('S1', 'S2', 'S3')},
                                  'max |gap|': float(gap.abs().max()),
                                  'human S1<S2<S3': bool(medians.loc['human', 'S1'] < medians.loc['human', 'S2'] < medians.loc['human', 'S3']),
                                  'overall Spearman vs saved': spearmanr(fit[key], COORDS[label], nan_policy='omit').statistic})
REGISTRATION = pd.DataFrame(registration_rows)
REGISTRATION.to_csv(output / 'species_registration_across_refits.csv', index=False)
display(REGISTRATION.round(3))

## 13 · Summary and pre-registered flags

In [ ]:
p1_fold = P1[P1.coordinate.str.endswith('fold-protected')].set_index(['coordinate', 'species', 'reference']).rho
p1w_fold = P1W[P1W.primary & P1W.coordinate.str.endswith('fold-protected')].groupby(['coordinate', 'species']).rho.mean()
absorb = ABSORPTION.set_index('coordinate')
disp = DISPLACEMENT[DISPLACEMENT.species.eq('human')].groupby('coordinate')['within-segment rho'].median()
summary = {
    'gene sets': set_counts.to_dict(),
    'reproduction': {k: (float(v) if not isinstance(v, bool) else v) for k, v in repro.items()},
    'P1 fold-protected': {' / '.join(k): float(v) for k, v in p1_fold.items()},
    'P1w fold-protected (primary references)': {' / '.join(k): float(v) for k, v in p1w_fold.items()},
    'P3a median gene-fold within-segment agreement': STABILITY.groupby('coordinate').rho.median().round(4).to_dict(),
    'positive controls (all) sign agreement': CONTROLS[CONTROLS.set.eq('all positive controls')]
        .set_index('coordinate')['sign agreement'].round(4).to_dict(),
    'P2': {c: {'median ratio': float(absorb.loc[c, 'median_ratio']), 'slope': float(absorb.loc[c, 'slope']),
               'human within-segment rho': float(disp.loc[c]),
               'material absorption flag': bool(absorb.loc[c, 'median_ratio'] < .85 or disp.loc[c] < .90)}
           for c in absorb.index},
    'P5 median within-segment agreement': P5_MEDIAN.round(4).to_dict(),
    'numerical floor median within-segment rho': FLOOR.groupby('coordinate')['within-segment rho'].median().round(4).to_dict(),
    'A3': {k: (v if isinstance(v, (str, bool)) else float(v)) for k, v in A3.items()},
    'A3b excess human sigma': sigma_ex,
    'A3b summary': json.loads(summary_noise.reset_index().to_json(orient='records')),
    'G6': json.loads(COUNT_SPLIT.to_json(orient='records')),
    'n_map T_spatial': MAP_STATS.mean(numeric_only=True).drop('map').round(4).to_dict(),
    'original split': SPLIT12,
    'registration max gap': REGISTRATION.set_index(['coordinate', 'refit'])['max |gap|'].round(3).unstack(0).to_dict(),
    'logic_version': NOTEBOOK_LOGIC_VERSION,
}
(output / 'summary.json').write_text(json.dumps(summary, indent=1, default=float))
print(json.dumps({k: summary[k] for k in ('P1 fold-protected', 'P1w fold-protected (primary references)', 'P2',
                                          'P5 median within-segment agreement', 'A3b excess human sigma')},
                 indent=1, default=float))